# Expert challenge — Your models in a mission

**Optional, outside the leaderboard.** Allow 30–45 minutes. You can edit this notebook: choose your hypotheses, settings, models and analyses. A coding agent can help you write code and train models; it is optional and does not determine the result.

**Measured mission: open the drawer with a learned policy, put away the cube, then close the drawer.** Everything happens in the same scene. Opening uses your RL policy or visual student; grasping, placement and closure remain programmed controllers. No fallback model hides a failure of your opening policy.

1. Measure the reference. 2. Reuse a model from notebook 01 or train a variant. 3. Measure your model. 4. Explain a failure and compare.

The standard measurement runs through `expert.sh`, separately from your exploration code. It checks the physical state, not an agent’s claims. Inspection calls earn no points. **Watch Isaac Sim during measurement.** Do not run an agent on the robot, reset the scene or use another notebook during this command. Resets are neither a hidden benchmark nor seed-paired episodes; three trials help diagnose behaviour, but do not establish statistical superiority.

In [ ]:
import os, sys, json, time, glob, subprocess, shlex
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RT = Path(os.environ.get("VINCI_RUNTIME", "/opt/vinci-workshop"))
os.environ["VINCI_WORKSPACE"] = str(ROOT)
sys.path[:0] = [str(ROOT), str(RT), str(RT / "lib")]
def run(args):
    env = dict(os.environ); env.pop("MPLBACKEND", None)
    with subprocess.Popen(args, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          stdin=subprocess.DEVNULL, text=True) as p:
        for line in p.stdout: print(line.rstrip(), flush=True)
        rc = p.wait()
    if rc: raise RuntimeError(f"Command failed ({rc}): {shlex.join(args)}. Fix the error before continuing.")
def reports():
    return {str(p) for p in (ROOT / "results/expert").glob("measure_*/report.json")}
def measure(reference, label):
    previous = reports()
    args = ["./expert.sh", "evaluate", "--provider", PROVIDER, "--episodes", str(EPISODES), "--label", label]
    if reference: args += ["--reference"]
    run(args)
    added = reports() - previous
    if len(added) != 1: raise RuntimeError("Expected exactly one new report; check results/expert")
    path = Path(added.pop()); result = json.loads(path.read_text())
    if result["status"] != "complete": raise RuntimeError(f"Incomplete measurement: {path}")
    return result
print("Workspace:", ROOT)

## 1. Choose a policy type and measure the reference

`visual`: two RGB-D cameras and joint readings; a student trained through imitation/DAgger.
`rl`: a PPO policy with privileged state, including the handle position, without images.

First compare two models **of the same type**. Three episodes provide an initial diagnostic. Increase the count to investigate further, using the same budget before and after. The reference is selected temporarily; your selection is then restored.

Allow approximately **70 seconds per mission**, or **3–4 minutes for three trials**. A result line appears after each episode; watch the robot in Isaac Sim while waiting.

In [ ]:
PROVIDER = "visual"  # "visual" or "rl"
EPISODES = 3
BEFORE = measure(reference=True, label="reference")

## 2. Reuse the guided notebook’s model, or train a variant

Models from notebook 01 remain in `results/`, even after the reference has been restored. To reuse one, skip training and specify its folder when selecting the model.

For training, change **one factor at a time** and write down your hypothesis. The visual student learns from the supplied teacher; your short RL training run does not automatically become its teacher. RAM and the other motions are not retrained here.

Initial settings: RL, 256 environments / 50 iterations; DAgger, 64 environments, 128 demonstrations, 2 rounds. The `0.7 / 0.35` perturbations match the guided notebook, rather than adding a new difficulty. A DAgger run of this size took about 4 min 15 s during preparation; duration depends on the settings. Follow training in the logs, not the Isaac window.

In [ ]:
RL = dict(num_envs=256, max_iterations=50)
DAGGER = dict(num_envs=64, bc_batches=2, round_batches=2, slip_prob=0.7, push_rad=0.35)
TRAIN_NEW = False  # True to train; False to reuse your model from notebook 01
RUN = ""           # example: results/train_dagger_20261005_... (use an actual path)
if TRAIN_NEW:
    prefix = "train_dagger_" if PROVIDER == "visual" else "train_rl_"
    existing = set((ROOT / "results").glob(prefix + "*"))
    if PROVIDER == "visual":
        args = ["./train.sh", "dagger", "--stages", "teacher_eval,bc,r1,r2,report", "--n_video", "1", "--perturb"]
        for k, v in DAGGER.items(): args += ["--" + k, str(v)]
    else:
        args = ["./train.sh", "rl"]
        for k, v in RL.items(): args += ["--" + k, str(v)]
    run(args)
    produced = set((ROOT / "results").glob(prefix + "*")) - existing
    if len(produced) != 1: raise RuntimeError("Cannot identify this training's output")
    new = produced.pop()
    if not (new / "policy/MODEL.json").exists(): raise RuntimeError("No exported checkpoint; do not continue")
    RUN = str(new.relative_to(ROOT))
else:
    for p in sorted((ROOT / "results").glob("train_*/policy/MODEL.json")):
        m = json.loads(p.read_text()); print(p.parent.parent.relative_to(ROOT), m["name"], m["sha256"][:12])

## 3. Explicitly select your checkpoint

Enter the selected folder below if `RUN` is empty. Execution stops without a compatible model. The fingerprint identifies the weights actually used; it does not prove that they are better.

In [ ]:
# RUN = "results/train_dagger_..."  # uncomment and enter the actual path if needed
if not RUN: raise ValueError("Choose RUN from the listed folders, or run your own training")
meta = json.loads((ROOT / RUN / "policy/MODEL.json").read_text())
expected = "open_drawer_student" if PROVIDER == "visual" else "open_drawer_rl"
if meta["name"] != expected: raise ValueError(f"{meta['name']} does not match {PROVIDER}")
run(["./train.sh", "select", RUN])
print("Selected:", meta["name"], meta["sha256"])

## 4. Run the mission again and compare

The same sequence runs with your model. **A failed opening remains a failure**, even if another policy could have rescued it. The mission succeeds only if the cube is put away and the drawer is closed. No agent supplies the verdict.

If you change parameters afterwards, repeat training, selection and measurement. For the initial comparison, keep the same provider and episode count.

In [ ]:
AFTER = measure(reference=False, label="personal")
for name, r in [("reference", BEFORE), ("personal", AFTER)]:
    print(name, "| opening", r["opening_successes"], "/", r["requested_episodes"], "| mission", r["mission_successes"], "/", r["requested_episodes"], "| model", r["model"]["sha256"][:12])
if BEFORE["provider"] != AFTER["provider"] or BEFORE["requested_episodes"] != AFTER["requested_episodes"]:
    print("Warning: different protocols; a direct comparison is not valid.")

## 5. Explore and present your result

You may edit the notebook, data budgets, training parameters and Python strategy, and write your own experiments. To obtain a comparable measurement, rerun the standard mission with `expert.sh`. Do not change its evaluator or the simulator files.

Show **your hypothesis, an observed failure, the checkpoint tested, before/after results and their limitations**. A carefully analysed lack of improvement is more useful than an unmeasured success claim. If every trial already succeeds, do not claim a higher success rate: investigate the required data budget, time or perception instead.

The standard mission isolates the effect of your opening policy. Custom compositions remain an exploration to demonstrate at the debrief, outside the leaderboard. Your coding agent may help you implement them, but must not act on the scene while measurement is running.

## Optional extension: learn to verify RAM insertion from images

This second problem is independent: the RAM stick is already in the gripper, in a different scene. The RAM student is supplied; here, **you train a visual classifier**, not the insertion policy.

The classifier receives only the two allowed images. Simulator ground truth supplies training labels and evaluates predictions. Validation examples use different seeds and remain separate from training data. “Before insertion” negatives are easy: good accuracy on this dataset does not establish that the classifier detects almost successful insertions.

Collecting 20 examples may take several minutes (about 13–20 s per insertion). This extension is optional. Do not run it during another measurement.

In [ ]:
# Optional: no data collection starts while COLLECT is False.
COLLECT = False
if COLLECT:
    run(["./expert_verifier.py", "collect", "--scene", "pc", "--train", "12", "--validation", "8"])
    run(["./expert_verifier.py", "fit", "--scene", "pc"])
    run(["./expert_verifier.py", "predict", "--scene", "pc"])